# Inferenza Layer Normativi

## Principio metodologico

I livelli della piramide normativa **emergono dai dati**.
L'algoritmo identifica cluster di atti che condividono la stessa **funzione normativa**,
indipendentemente dall'argomento trattato.

Punto critico: due regolamenti che trattano lo stesso argomento ma hanno funzioni diverse
(uno quadro, uno tecnico-operativo) devono finire in cluster diversi. Per questo non usiamo
embedding del testo completo (che cattura il dominio tematico), ma **feature funzionali**
che catturano il ruolo normativo.

## Feature utilizzate

| Gruppo | Cosa cattura | Come si estrae |
|---|---|---|
| **Funzionali** | Ruolo normativo: chi adotta, con quale procedura, che obbligo crea | Regex + LLM (domande chiuse); `primary_obligation` separata per fonte (regex ×2, LLM ×0.5) |
| **Strutturali** | Posizione nella gerarchia della rete di citazioni (indegree, outdegree, citation_ratio, pagerank) | NetworkX sul grafo |
| **Semantiche ridotte** | Sfumature linguistiche residue non catturate da regex | Embedding su titolo + 200c preambolo |

## Pipeline

```
1. Feature strutturali dal grafo
2. Feature funzionali (regex → LLM per i campi mancanti)
3. Embedding su testo breve
4. Normalizzazione e concatenazione
5. UMAP: riduzione dimensionale
6. HDBSCAN: clustering — il numero di layer emerge dai dati
7. Ordinamento gerarchico dei cluster
8. Purezza, entropia, level_span per ogni atto
9. Validazione post-hoc
```

## Input / Output
- **Input**: `nodes_focal_texts.csv`, `edges_focal.csv`
- **Output**: `nodes_focal_layers.csv`

## 0. Setup e Parametri

In [27]:
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import numpy as np
import re, os, sys, json, time

sys.path.append('..')
from config_golden_power import MATERIA_NAME

# ── Percorsi ─────────────────────────────────────────────────────────────────
output_path     = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file      = os.path.join(output_path, 'nodes_focal_texts.csv')
edges_file      = os.path.join(output_path, 'edges_focal.csv')
output_file     = os.path.join(output_path, 'nodes_focal_layers.csv')
checkpoint_file = os.path.join(output_path, 'functional_features_checkpoint.csv')

# ── Parametri LLM ────────────────────────────────────────────────────────────
LLM_MODEL        = 'deepseek-chat'
LLM_DELAY        = 0.3
LLM_MAX_RETRIES  = 3
CHECKPOINT_EVERY = 50

# ── Parametri UMAP ───────────────────────────────────────────────────────────
UMAP_N_COMPONENTS = 15
UMAP_N_NEIGHBORS  = 20
UMAP_MIN_DIST     = 0.05
UMAP_RANDOM_STATE = 42

# ── Parametri HDBSCAN ────────────────────────────────────────────────────────
# min_cluster_size: dimensione minima per essere considerato un layer.
# 400 → 4-6 layer macro, più leggibili e allineati con Lamfalussy.
# Diminuire → più layer, più fini. Aumentare → meno layer, più generali.
HDBSCAN_MIN_CLUSTER_SIZE = 200
HDBSCAN_MIN_SAMPLES      = 25

# ── Soglie interpretazione ───────────────────────────────────────────────────
PURITY_THRESHOLD     = 0.70   # sotto questa soglia: atto ibrido
MEMBERSHIP_THRESHOLD = 0.15   # soglia per contare un layer nel level_span
SPAN_THRESHOLD       = 2      # level_span >= 2: potenzialmente patologico

# ── Pesi ordinamento gerarchico ───────────────────────────────────────────────
# Determinano quale segnale pesa di più nel decidere quale cluster è L1.
# I due pesi devono sommare a 1.0.
# RATIO:    citation_ratio medio del cluster (atti apicali citati-da-molti / cita-pochi)
# RECEIVED: citazioni inter-cluster ricevute (apicale = tutti lo citano)
# Cambiare questi valori e ri-eseguire la cella 9 non richiede di rieseguire il clustering.
HIERARCHY_W_RATIO    = 0.6
HIERARCHY_W_RECEIVED = 0.4
assert abs(HIERARCHY_W_RATIO + HIERARCHY_W_RECEIVED - 1.0) < 1e-9, "I pesi devono sommare a 1.0"

# ── Modello embedding ─────────────────────────────────────────────────────────
# all-mpnet-base-v2: 768d → PCA 50d. Più potente di MiniLM ma più lento (~2 min).
# Nota: anche con testo breve (200c) mpnet può catturare sfumature tematiche;
# questo è accettabile perché il suo peso nella feature matrix è bilanciato
# dalle feature funzionali e strutturali (×2).
EMBEDDING_MODEL = 'all-mpnet-base-v2'

print(f"Input:  {input_file}")
print(f"Output: {output_file}")
print(f"Hierarchy weights — ratio: {HIERARCHY_W_RATIO}  received: {HIERARCHY_W_RECEIVED}")

Input:  ..\data\output\golden_power\nodes_focal_texts.csv
Output: ..\data\output\golden_power\nodes_focal_layers.csv
Hierarchy weights — ratio: 0.6  received: 0.4


## 1. Caricamento Dati e Feature Strutturali

Il grafo ha **29 tipi di relazione** con semantica normativa precisa.
Vengono sfruttati costruendo:
- **Grafo pesato** per tipo (BASED_ON=3, IMPLEMENTS=2.5, CITES=0.3, ecc.) → PageRank e HITS gerarchici
- **Contatori per tipo** (`based_on_received`, `amends_sent`, ecc.) → feature dirette
- **Feature aggregate** per famiglia (`hierarchical_authority`)
- **Normalizzazione temporale**: ogni feature viene divisa per la media degli atti dello stesso anno,
  eliminando il bias per cui atti vecchi appaiono più centrali solo per anzianità
- **HITS**: decompone ogni nodo in authority (apicale) e hub (subordinato)
- **K-core**: posizione topologica nel nucleo della rete, indipendente dal tempo

In [28]:
import networkx as nx

nodes = pd.read_csv(input_file)
edges = pd.read_csv(edges_file)

nodes = nodes.drop_duplicates(subset=['Id'])
print(f"Nodi:  {len(nodes)}")
print(f"Archi: {len(edges)}")

# Adatta nomi colonne archi
src_col  = 'Source' if 'Source' in edges.columns else edges.columns[0]
tgt_col  = 'Target' if 'Target' in edges.columns else edges.columns[1]
type_col = 'Type'   if 'Type'   in edges.columns else (':TYPE' if ':TYPE' in edges.columns else None)

if type_col:
    print(f"\nTipi di relazione presenti ({edges[type_col].nunique()}):")
    print(edges[type_col].value_counts().to_string())

# ── Famiglie di tipo per valore gerarchico ─────────────────────────────────────
# HIERARCHICAL_DOWN: A cita B come base/fondamento → B è sopra A
HIERARCHICAL_DOWN = {'BASED_ON', 'IMPLEMENTS', 'ADOPTS', 'PARTIALLY_ADOPTS'}
# LATERAL: stessa fascia gerarchica (modifica, corregge, completa)
LATERAL           = {'AMENDS', 'CORRECTS', 'COMPLETES', 'ADDS_TO',
                     'DOES_INSERTION', 'DOES_DELETION', 'DOES_REPLACEMENT',
                     'EXTENDS_APPLICATION', 'EXTENDS_VALIDITY', 'REPLACES',
                     'DOES_REPEAL', 'REESTABLISHES'}
# ABROGATION: A abroga B → stesso livello, A più recente
ABROGATION        = {'REPEALS', 'IMPLICITLY_REPEALS'}
# SOFT: segnale debole, direzione gerarchica incerta
SOFT              = {'CITES', 'RELATED_TO', 'RELATED_QUESTION_TO',
                     'INFLUENCES', 'DEROGATES', 'SUSPENDS', 'PARTIALLY_SUSPENDS',
                     'PROPOSES_TO_AMEND', 'DEFERS_APPLICATION', 'INCORPORATES'}
# INTERPRETIVE: solo sentenze CGUE
INTERPRETIVE      = {'INTERPRETES_AUTHORITATIVELY'}

# ── Pesi per grafo pesato ──────────────────────────────────────────────────────
# Peso proporzionale alla certezza gerarchica del segnale
EDGE_WEIGHTS = {
    'BASED_ON':                   3.0,
    'IMPLEMENTS':                 2.5,
    'ADOPTS':                     2.0,
    'PARTIALLY_ADOPTS':           2.0,
    'AMENDS':                     1.0,
    'CORRECTS':                   0.8,
    'REPEALS':                    1.0,
    'IMPLICITLY_REPEALS':         0.8,
    'COMPLETES':                  0.8,
    'DOES_REPLACEMENT':           0.8,
    'DOES_INSERTION':             0.5,
    'DOES_DELETION':              0.5,
    'DOES_REPEAL':                0.5,
    'EXTENDS_APPLICATION':        0.5,
    'EXTENDS_VALIDITY':           0.5,
    'REPLACES':                   0.8,
    'DEROGATES':                  0.5,
    'INTERPRETES_AUTHORITATIVELY':1.5,
    'CITES':                      0.3,
}
DEFAULT_WEIGHT = 0.3

# ── Costruzione grafi ─────────────────────────────────────────────────────────
# G_unweighted: per k-core e feature di conteggio
# G_weighted:   per PageRank e HITS gerarchici
G = nx.DiGraph()
G.add_nodes_from(nodes['Id'].tolist())

G_w = nx.DiGraph()
G_w.add_nodes_from(nodes['Id'].tolist())

for _, row in edges.iterrows():
    s, t = row[src_col], row[tgt_col]
    et   = row[type_col] if type_col else 'CITES'
    w    = EDGE_WEIGHTS.get(et, DEFAULT_WEIGHT)
    G.add_edge(s, t, edge_type=et)
    G_w.add_edge(s, t, weight=w)

# ── Feature di conteggio per tipo ─────────────────────────────────────────────
print("\nCalcolo feature strutturali tipate...")

# Contatori per nodo
type_counts = {nid: {} for nid in nodes['Id']}

for _, row in edges.iterrows():
    s, t = row[src_col], row[tgt_col]
    et   = row[type_col] if type_col else 'CITES'
    # ricevuto (in): s viene 'raggiunto' da t? No: in un arco s→t, s EMETTE, t RICEVE
    if t in type_counts:
        key_in = f"{et.lower()}_received"
        type_counts[t][key_in] = type_counts[t].get(key_in, 0) + 1
    if s in type_counts:
        key_out = f"{et.lower()}_sent"
        type_counts[s][key_out] = type_counts[s].get(key_out, 0) + 1

counts_df = pd.DataFrame.from_dict(type_counts, orient='index').fillna(0)
counts_df.index.name = 'Id'
counts_df = counts_df.reset_index()

nodes = nodes.merge(counts_df, on='Id', how='left')
# Riempi eventuali NaN (nodi senza archi)
count_cols = [c for c in nodes.columns if c.endswith('_received') or c.endswith('_sent')]
nodes[count_cols] = nodes[count_cols].fillna(0)

# ── Feature aggregate per famiglia ─────────────────────────────────────────────
# Autorità gerarchica: quanti atti si fondano/implementano su questo
nodes['hierarchical_authority'] = sum(
    nodes.get(f"{et.lower()}_received", pd.Series(0, index=nodes.index))
    for et in HIERARCHICAL_DOWN
)
# Subordinazione gerarchica: su quanti atti si fonda/implementa
nodes['hierarchical_subordination'] = sum(
    nodes.get(f"{et.lower()}_sent", pd.Series(0, index=nodes.index))
    for et in HIERARCHICAL_DOWN
)
# Attività laterale: modifiche emesse/ricevute
nodes['lateral_activity'] = sum(
    nodes.get(f"{et.lower()}_received", pd.Series(0, index=nodes.index)) +
    nodes.get(f"{et.lower()}_sent",     pd.Series(0, index=nodes.index))
    for et in LATERAL
)
# Ratio gerarchico: atti che si fondano su di me / atti su cui mi fondo
nodes['hierarchical_ratio'] = (
    nodes['hierarchical_authority'] /
    (nodes['hierarchical_subordination'] + 1)
)

# ── Indegree / outdegree generici (backward compat) ────────────────────────────
indegree  = dict(G.in_degree())
outdegree = dict(G.out_degree())
nodes['indegree']       = nodes['Id'].map(indegree).fillna(0)
nodes['outdegree']      = nodes['Id'].map(outdegree).fillna(0)
nodes['citation_ratio'] = nodes['indegree'] / (nodes['outdegree'] + 1)

# ── PageRank su grafo pesato ───────────────────────────────────────────────────
pagerank_w = nx.pagerank(G_w, alpha=0.85, max_iter=300, weight='weight')
nodes['pagerank'] = nodes['Id'].map(pagerank_w).fillna(0)

# ── HITS su grafo pesato ───────────────────────────────────────────────────────
# hub_score:       quanto questo atto punta ad authority importanti (subordinazione)
# authority_score: quanto questo atto è puntato da hub importanti (apicalità)
try:
    hits_hub, hits_auth = nx.hits(G_w, max_iter=500, normalized=True)
    nodes['hits_authority'] = nodes['Id'].map(hits_auth).fillna(0)
    nodes['hits_hub']       = nodes['Id'].map(hits_hub).fillna(0)
    # Ratio HITS: alta authority + basso hub → apicale
    nodes['hits_ratio'] = nodes['hits_authority'] / (nodes['hits_hub'] + 1e-9)
    print("HITS: ok")
except nx.PowerIterationFailedConvergence:
    print("HITS: non converge — impostati a 0")
    nodes['hits_authority'] = 0.0
    nodes['hits_hub']       = 0.0
    nodes['hits_ratio']     = 0.0

# ── K-core su grafo non diretto (posizione topologica) ────────────────────────
G_und = G.to_undirected()
G_und.remove_edges_from(nx.selfloop_edges(G_und))  # k-core non supporta self-loop
core_number = nx.core_number(G_und)
nodes['kcore'] = nodes['Id'].map(core_number).fillna(0)

# ── Normalizzazione temporale delle feature strutturali ───────────────────────
# Problema: indegree/pagerank dipendono dall'anzianità dell'atto.
# Un regolamento del 1990 ha indegree alto perché ha avuto 30 anni per essere citato,
# non necessariamente perché è più apicale di uno del 2020.
# Soluzione: dividere per la media degli atti dello stesso anno.
# Il risultato misura 'quanto è citato rispetto ai suoi contemporanei'.

if 'Year' in nodes.columns:
    year_col = 'Year'
elif 'year' in nodes.columns:
    year_col = 'year'
else:
    year_col = None

if year_col:
    for feat in ['indegree', 'pagerank', 'hits_authority', 'hierarchical_authority']:
        year_mean = nodes.groupby(year_col)[feat].transform('mean').replace(0, np.nan)
        nodes[f"{feat}_norm"] = (nodes[feat] / year_mean).fillna(1.0)
    print("Normalizzazione temporale: ok")
else:
    print("[SKIP] Colonna anno non trovata — normalizzazione temporale disabilitata")
    for feat in ['indegree', 'pagerank', 'hits_authority', 'hierarchical_authority']:
        nodes[f"{feat}_norm"] = nodes[feat]

# ── Riepilogo ─────────────────────────────────────────────────────────────────
print()
print("Feature strutturali calcolate:")
struct_report = [
    ('indegree',                  'archi totali ricevuti'),
    ('outdegree',                 'archi totali emessi'),
    ('citation_ratio',            'indegree / (outdegree+1)'),
    ('pagerank',                  'PageRank su grafo pesato per tipo'),
    ('hits_authority',            'HITS authority (apicalità)'),
    ('hits_hub',                  'HITS hub (subordinazione)'),
    ('hits_ratio',                'authority / hub'),
    ('kcore',                     'k-core (posizione topologica)'),
    ('hierarchical_authority',    'BASED_ON/IMPLEMENTS ricevuti'),
    ('hierarchical_subordination','BASED_ON/IMPLEMENTS emessi'),
    ('hierarchical_ratio',        'authority ger. / subordinazione ger.'),
    ('indegree_norm',             'indegree normalizzato per anno'),
    ('pagerank_norm',             'pagerank normalizzato per anno'),
    ('hits_authority_norm',       'hits_authority normalizzato per anno'),
    ('hierarchical_authority_norm','hierarchical_authority norm. per anno'),
    ('based_on_received',         'quanti atti si fondano giuridicamente su un atto'),
    ('implements_received',       'quanti atti implementano un atto'),
    ('amends_received',           'quante volte un atto è stato modificato'),
    ('amends_sent',               'quanti atti un atto ha modificato'),
    ('repeals_sent',              'quanti atti un atto ha abrogato'),
]
for col, desc in struct_report:
    if col in nodes.columns:
        print(f"  {col:<35} mean={nodes[col].mean():.4f}  max={nodes[col].max():.4f}")

# Feature di conteggio per tipo più frequenti
print()
print("Top feature di conteggio per tipo (mean > 0.01):")
for c in sorted(count_cols):
    m = nodes[c].mean()
    if m > 0.01:
        print(f"  {c:<40} mean={m:.3f}  max={nodes[c].max():.0f}")

Nodi:  4416
Archi: 25743

Tipi di relazione presenti (17):
Type
CITES                  17964
BASED_ON                4385
AMENDS                  1385
CORRECTS                1034
REPEALS                  351
IMPLICITLY_REPEALS       155
DOES_REPLACEMENT         127
COMPLETES                107
EXTENDS_VALIDITY          71
DOES_DELETION             54
DOES_INSERTION            51
DEROGATES                 27
REPLACES                  10
DOES_REPEAL                9
EXTENDS_APPLICATION        7
IMPLEMENTS                 5
RELATED_TO                 1

Calcolo feature strutturali tipate...
HITS: ok
Normalizzazione temporale: ok

Feature strutturali calcolate:
  indegree                            mean=5.6311  max=298.0000
  outdegree                           mean=5.6311  max=73.0000
  citation_ratio                      mean=2.8537  max=298.0000
  pagerank                            mean=0.0002  max=0.0061
  hits_authority                      mean=0.0002  max=0.2497
  hits_hub        

## 2. Feature Funzionali via Regex

Estrae autore, procedura, base TFUE e tipo di obbligo dai pattern testuali
inequivocabili. Il LLM (Step 3) riempirà solo i campi rimasti vuoti.

In [29]:
def extract_regex_features(title, preamble):
    title_s    = str(title)    if pd.notna(title)    else ''
    preamble_s = str(preamble) if pd.notna(preamble) else ''
    # Usa titolo + primi 600 caratteri del preambolo:
    # procedura e base giuridica sono sempre nelle prime righe
    title_u = title_s.upper()
    text_u  = f"{title_s} {preamble_s[:600]}".upper()

    f = {'author': None, 'procedure': None,
         'tfeu_article': None, 'primary_obligation': None,
         'has_technical_annex': False}

    # ── Autore istituzionale (dal più specifico al più generale) ──────────────
    if re.search(r'JUDGMENT OF THE COURT|ORDER OF THE COURT|OPINION OF THE COURT', title_u):
        f['author'] = 'Court'
    elif re.search(r'COMMISSION DELEGATED REGULATION|DELEGATED REGULATION\s*\(EU\)', title_u):
        f['author'] = 'Commission_delegated'
    elif re.search(r'COUNCIL IMPLEMENTING REGULATION|COUNCIL IMPLEMENTING DECISION', title_u):
        f['author'] = 'Council'
    elif re.search(r'COMMISSION IMPLEMENTING REGULATION|IMPLEMENTING REGULATION\s*\(EU\)|COMMISSION IMPLEMENTING DECISION|IMPLEMENTING DECISION\s*\(EU\)', title_u):
        f['author'] = 'Commission_implementing'
    elif re.search(r'REGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL|DIRECTIVE OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL|DECISION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL', title_u):
        f['author'] = 'EP_Council'
    elif re.search(r'COUNCIL REGULATION|COUNCIL DIRECTIVE|COUNCIL DECISION|COUNCIL FRAMEWORK DECISION', title_u):
        f['author'] = 'Council'
    elif re.search(r'COMMISSION RECOMMENDATION|COMMISSION COMMUNICATION|COMMISSION DECISION|COMMISSION REGULATION', title_u):
        f['author'] = 'Commission_other'
    elif re.search(r'^EUR-LEX\s*-\s*\d', title_u):
        # Titolo non recuperato (pagina EUR-Lex grezza) — lascia None, andrà all'LLM
        f['author'] = None
    elif re.search(r'TREATY|TFEU|TREATY ON THE FUNCTIONING', title_u):
        f['author'] = 'Treaty'

    # ── Procedura adottiva — titolo + preambolo ───────────────────────────────
    if re.search(r'ORDINARY LEGISLATIVE PROCEDURE|CO-DECISION PROCEDURE', text_u):
        f['procedure'] = 'ordinary_legislative'
    elif re.search(r'PURSUANT TO ARTICLE 29[01]|DELEGATED BY|EMPOWERED BY ARTICLE', text_u):
        f['procedure'] = 'delegated_implementing'
    elif re.search(r'ARTICLE 258|ARTICLE 260|INFRINGEMENT PROCEEDINGS|HAS FAILED TO FULFIL', text_u):
        f['procedure'] = 'infringement'
    elif re.search(r'HEREBY RECOMMENDS|NON-BINDING|SHOULD BE UNDERSTOOD', text_u):
        f['procedure'] = 'recommendation'
    elif f['author'] == 'Court':
        f['procedure'] = 'judgment'
    elif f['author'] == 'Treaty':
        f['procedure'] = 'treaty'

    # ── Base giuridica TFUE ───────────────────────────────────────────────────
    m = re.search(r'ARTICLE\s+(\d+)\s*(?:THEREOF|TFEU|TEEU|OF THE TREATY ON THE FUNCTIONING)', text_u)
    if m:
        f['tfeu_article'] = int(m.group(1))

    # ── Tipo di obbligo primario ──────────────────────────────────────────────
    if f['author'] == 'Court':
        f['primary_obligation'] = 'judicial_ruling'
    elif re.search(r'SHALL BE PROHIBITED|IS HEREBY PROHIBITED', text_u):
        f['primary_obligation'] = 'prohibition'
    elif re.search(r'THE COURT RULES|THE ACTION IS DISMISSED|ANNULS THE|DECLARES THAT', text_u):
        f['primary_obligation'] = 'judicial_ruling'
    elif re.search(r'HEREBY RECOMMENDS|SHOULD(?!\s+ENSURE)|IS ENCOURAGED', text_u[:300]):
        f['primary_obligation'] = 'recommendation'
    elif re.search(r'SHALL BE CALCULATED|THE FORM SET OUT|AS SET OUT IN THE ANNEX|STANDARD FORM|TECHNICAL SPECIFICATION', text_u):
        f['primary_obligation'] = 'technical_standard'
    elif re.search(r'MEMBER STATES SHALL|IS HEREBY ESTABLISHED|SHALL ENSURE|SHALL APPLY', text_u):
        f['primary_obligation'] = 'obligation'

    # ── Allegati tecnici ──────────────────────────────────────────────────────
    f['has_technical_annex'] = bool(
        re.search(r'STANDARD FORM|THE LIST SET OUT|AS SET OUT IN ANNEX|TEMPLATE|TECHNICAL SPECIFICATIONS', text_u)
    )

    return f


print("Estrazione feature via regex...")
regex_results = nodes.apply(lambda r: extract_regex_features(r.get('title'), r.get('preamble')), axis=1)
regex_df = pd.DataFrame(list(regex_results))
for col in regex_df.columns:
    nodes[f'rx_{col}'] = regex_df[col].values

print("Copertura regex:")
for col in ['rx_author','rx_procedure','rx_tfeu_article','rx_primary_obligation']:
    n = nodes[col].notna().sum()
    print(f"  {col:<30} {n:>5} / {len(nodes)}  ({n/len(nodes)*100:.1f}%)")

print("\nDistribuzione author (regex):")
print(nodes['rx_author'].value_counts(dropna=False).to_string())

Estrazione feature via regex...
Copertura regex:
  rx_author                       2214 / 4416  (50.1%)
  rx_procedure                     665 / 4416  (15.1%)
  rx_tfeu_article                  980 / 4416  (22.2%)
  rx_primary_obligation            190 / 4416  (4.3%)

Distribuzione author (regex):
rx_author
NaN                        2202
Council                     843
Commission_implementing     490
Commission_other            435
Commission_delegated        261
Court                       136
Treaty                       40
EP_Council                    9


## 3. Completamento Feature via LLM

Il LLM viene chiamato **solo** per i nodi in cui regex non ha trovato
`author` o `procedure`. Usa domande a risposta chiusa: non classifica,
risponde a domande precise su testo breve.

In [30]:
from openai import OpenAI, RateLimitError
client = OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1" 
)

SYSTEM_PROMPT = """Sei un esperto di tecnica legislativa UE.
Rispondi ESCLUSIVAMENTE con JSON valido, senza testo aggiuntivo, senza backtick."""

def llm_extract(title, preamble):
    title_s    = str(title)    if pd.notna(title)    else ''
    preamble_s = str(preamble) if pd.notna(preamble) else ''
    if not title_s and not preamble_s:
        return None, 'no_text'

    prompt = f"""Analizza questo atto normativo UE.

TESTO:
{title_s}
{preamble_s[:500]}

Rispondi SOLO con questo JSON (scegli tra i valori indicati, null se non determinabile):
{{
  "author": "EP_Council / Council / Commission_delegated / Commission_implementing / Commission_other / Court / Treaty / Other",
  "procedure": "ordinary_legislative / delegated_implementing / infringement / recommendation / judgment / treaty / other",
  "tfeu_article": numero intero o null,
  "primary_obligation": "prohibition / obligation / technical_standard / recommendation / judicial_ruling / other"
}}"""

    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp    = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role':'system','content':SYSTEM_PROMPT},
                          {'role':'user','content':prompt}],
                temperature=0, max_tokens=200,
            )
            content = resp.choices[0].message.content.strip()
            content = content.replace('```json','').replace('```','').strip()
            parsed  = json.loads(content)
            art = parsed.get('tfeu_article')
            if art is not None:
                try:    parsed['tfeu_article'] = int(str(art).strip())
                except: parsed['tfeu_article'] = None
            return parsed, 'ok'
        except json.JSONDecodeError:
            if attempt < LLM_MAX_RETRIES - 1: time.sleep(1)
        except RateLimitError:
            time.sleep(30)
        except Exception as e:
            print(f"  ERRORE: {type(e).__name__}: {e}")
            return None, 'error'


In [ ]:
# Test
test_row = nodes[nodes['Label'] == '32019R0452'].iloc[0]
result, status = llm_extract(test_row.get('title'), test_row.get('preamble'))
print(f"Test LLM su 32019R0452: {status}")
print(json.dumps(result, indent=2))

In [31]:
# Nodi da completare: mancano author O procedure
needs_llm = nodes[
    (nodes['rx_author'].isna() | nodes['rx_procedure'].isna()) &
    (nodes['title'].notna() | nodes['preamble'].notna())
].copy()

CHKPT_COLS = ['Id','llm_author','llm_procedure','llm_tfeu_article','llm_primary_obligation','llm_status']
if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi già processati")
else:
    checkpoint   = pd.DataFrame(columns=CHKPT_COLS)
    already_done = set()

nodes_todo = needs_llm[~needs_llm['Id'].isin(already_done)]
print(f"Da completare con LLM: {len(nodes_todo)}")
print(f"Tempo stimato: ~{len(nodes_todo) * LLM_DELAY / 60:.0f} minuti")

results = []
n_ok = n_err = 0

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    parsed, status = llm_extract(row.get('title'), row.get('preamble'))
    if status == 'ok': n_ok += 1
    else:              n_err += 1

    results.append({
        'Id':                     row['Id'],
        'llm_author':             parsed.get('author')             if parsed else None,
        'llm_procedure':          parsed.get('procedure')          if parsed else None,
        'llm_tfeu_article':       parsed.get('tfeu_article')       if parsed else None,
        'llm_primary_obligation': parsed.get('primary_obligation') if parsed else None,
        'llm_status':             status,
    })

    if (i+1) % 10 == 0 or (i+1) == len(nodes_todo):
        print(f"  [{i+1:>4}/{len(nodes_todo)}] {(i+1)/len(nodes_todo)*100:5.1f}%  ok:{n_ok}  err:{n_err}")

    if (i+1) % CHECKPOINT_EVERY == 0:
        batch = pd.DataFrame(results)
        chkpt = pd.concat([checkpoint, batch]).drop_duplicates('Id')
        chkpt.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint ({len(chkpt)} totali)")

    time.sleep(LLM_DELAY)

if results:
    batch = pd.DataFrame(results)
    pd.concat([checkpoint, batch]).drop_duplicates('Id').to_csv(checkpoint_file, index=False)

print(f"\nCompletato. OK: {n_ok}  Errori: {n_err}")

Checkpoint trovato: 4495 nodi già processati
Da completare con LLM: 0
Tempo stimato: ~0 minuti

Completato. OK: 0  Errori: 0


## 3b. Valutazione Accuratezza LLM

Il regex è il riferimento: per i nodi in cui regex ha trovato `author` con certezza
ma LLM è stato chiamato comunque (perché mancava `procedure`), possiamo confrontare
`rx_author` (ground truth deterministico) con `llm_author` (risposta LLM).

Stesso ragionamento per `procedure` e `primary_obligation`.

In [32]:
# ── Valutazione accuratezza LLM su campo author ───────────────────────────────
# Campione di validazione: nodi dove ENTRAMBI hanno un valore
# (regex ha trovato → ground truth; LLM è stato chiamato → predizione)

if not os.path.exists(checkpoint_file):
    print("[SKIP] Checkpoint non trovato. Eseguire prima la cella 3.")
else:
    llm_check = pd.read_csv(checkpoint_file)
    # Unisce con regex results (già calcolate in cella 2)
    eval_df = nodes[['Id','rx_author','rx_procedure','rx_primary_obligation']].merge(
        llm_check[['Id','llm_author','llm_procedure','llm_primary_obligation','llm_status']],
        on='Id', how='inner'
    )
    eval_df = eval_df[eval_df['llm_status'] == 'ok']

    print(f"Nodi con risposta LLM valida nel checkpoint: {len(eval_df)}")
    print()

    def accuracy_report(field_regex, field_llm, label):
        """Calcola accuratezza LLM dove regex ha ground truth."""
        both = eval_df[eval_df[field_regex].notna() & eval_df[field_llm].notna()].copy()
        if len(both) == 0:
            print(f"{label}: nessun caso con entrambi i valori — impossibile valutare")
            return
        # Normalizzazione case-insensitive
        match = (both[field_regex].str.lower().str.strip() ==
                 both[field_llm].str.lower().str.strip())
        acc = match.mean()
        n   = len(both)
        print(f"{label}")
        print(f"  Campione di confronto: {n} nodi")
        print(f"  Accordo regex ↔ LLM:  {match.sum()} / {n}  ({acc*100:.1f}%)")
        if acc >= 0.90:
            print(f"  ✓ Affidabilità alta   (≥90%)")
        elif acc >= 0.75:
            print(f"  ~ Affidabilità media  (75-90%) — usare con cautela")
        else:
            print(f"  ✗ Affidabilità bassa  (<75%) — considerare peso ridotto nella feature matrix")
        # Errori più frequenti
        errors = both[~match][[field_regex, field_llm]]
        if len(errors) > 0:
            print(f"  Discordanze più frequenti:")
            top_errors = (errors.groupby([field_regex, field_llm])
                          .size().sort_values(ascending=False).head(5))
            for (r, l), cnt in top_errors.items():
                print(f"    regex={r:<30}  llm={l:<30}  ({cnt}x)")
        print()

    accuracy_report('rx_author',             'llm_author',             'AUTHOR')
    accuracy_report('rx_procedure',           'llm_procedure',           'PROCEDURE')
    accuracy_report('rx_primary_obligation',  'llm_primary_obligation',  'PRIMARY_OBLIGATION')

    # ── Nota metodologica sul campione ───────────────────────────────────────────
    print("Note metodologiche:")
    print("  - Il campione include solo nodi dove LLM è stato chiamato (rx_author o rx_procedure mancante).")
    print("  - Non è un campione casuale del corpus: è bias verso atti con struttura testuale meno standard.")
    print("  - L'accuratezza reale sui nodi LLM-only (senza rx ground truth) potrebbe differire.")
    print("  - Per author: il regex copre pattern inequivocabili → alta concordanza attesa (>90%).")
    print("  - Per primary_obligation: il regex copre <2% dei casi → campione di confronto molto piccolo.")

Nodi con risposta LLM valida nel checkpoint: 3505

AUTHOR
  Campione di confronto: 1978 nodi
  Accordo regex ↔ LLM:  1880 / 1978  (95.0%)
  ✓ Affidabilità alta   (≥90%)
  Discordanze più frequenti:
    regex=Commission_other                llm=Commission_implementing         (42x)
    regex=Council                         llm=EP_Council                      (30x)
    regex=Council                         llm=Commission_other                (9x)
    regex=Council                         llm=Commission_implementing         (5x)
    regex=Treaty                          llm=Council                         (4x)

PROCEDURE
  Campione di confronto: 429 nodi
  Accordo regex ↔ LLM:  423 / 429  (98.6%)
  ✓ Affidabilità alta   (≥90%)
  Discordanze più frequenti:
    regex=treaty                          llm=other                           (4x)
    regex=treaty                          llm=ordinary_legislative            (2x)

PRIMARY_OBLIGATION
  Campione di confronto: 54 nodi
  Accordo regex ↔ 

## 4. Merge Feature Finali

Regex ha priorità (deterministico). LLM riempie solo i campi rimasti vuoti.

`primary_obligation` viene tenuta **separata per fonte**: `primary_obligation_regex`
e `primary_obligation_llm`. Nella feature matrix riceveranno pesi diversi (×2 vs ×0.5)
perché la copertura regex è solo 1.4% — assegnare lo stesso peso sarebbe amplificare
il rumore LLM con la stessa forza del segnale deterministico.

In [33]:
if os.path.exists(checkpoint_file):
    llm_df = pd.read_csv(checkpoint_file)
    nodes  = nodes.merge(
        llm_df[['Id','llm_author','llm_procedure','llm_tfeu_article','llm_primary_obligation']],
        on='Id', how='left'
    )
else:
    for c in ['llm_author','llm_procedure','llm_tfeu_article','llm_primary_obligation']:
        nodes[c] = None

# Regex vince, LLM riempie i buchi
nodes['author']             = nodes['rx_author'].fillna(nodes['llm_author'])
nodes['procedure']          = nodes['rx_procedure'].fillna(nodes['llm_procedure'])
nodes['tfeu_article']       = nodes['rx_tfeu_article'].fillna(nodes['llm_tfeu_article'])
nodes['has_technical_annex'] = nodes['rx_has_technical_annex'].fillna(False)

# primary_obligation: mantiene le due fonti SEPARATE
# - primary_obligation_regex: deterministico, alta affidabilità, copertura 1.4%
# - primary_obligation_llm:   LLM, affidabilità media, copertura ~60%
# Riceveranno pesi diversi nella feature matrix (×2 vs ×0.5).
# La colonna unificata 'primary_obligation' è mantenuta solo per display/export.
nodes['primary_obligation_regex'] = nodes['rx_primary_obligation']
nodes['primary_obligation_llm']   = nodes['llm_primary_obligation']
nodes['primary_obligation']       = nodes['rx_primary_obligation'].fillna(nodes['llm_primary_obligation'])

print("Copertura feature funzionali (dopo merge):")
for col in ['author','procedure','tfeu_article',
            'primary_obligation','primary_obligation_regex','primary_obligation_llm']:
    n = nodes[col].notna().sum()
    src = '← regex+llm unified' if col == 'primary_obligation' else \
          '← deterministico'     if col == 'primary_obligation_regex' else \
          '← LLM only'           if col == 'primary_obligation_llm' else ''
    print(f"  {col:<30} {n:>5} / {len(nodes)}  ({n/len(nodes)*100:.1f}%)  {src}")

print("\nDistribuzione author:")
print(nodes['author'].value_counts(dropna=False).to_string())

Copertura feature funzionali (dopo merge):
  author                          3741 / 4416  (84.7%)  
  procedure                       3741 / 4416  (84.7%)  
  tfeu_article                    1851 / 4416  (41.9%)  
  primary_obligation              3641 / 4416  (82.5%)  ← regex+llm unified
  primary_obligation_regex         190 / 4416  (4.3%)  ← deterministico
  primary_obligation_llm          3505 / 4416  (79.4%)  ← LLM only

Distribuzione author:
author
Council                    1203
Commission_other            734
EP_Council                  684
NaN                         675
Commission_implementing     501
Commission_delegated        268
Court                       207
Other                       100
Treaty                       44


## 4.1 Fallback sul CELEX

In [34]:
# Terzo livello di fallback: inferisce author e procedure direttamente dal codice CELEX, deterministicamente, senza testo né API.
# SOLO quando l'identificazione è univoca ed affidabile
#
# Struttura CELEX: {settore}{anno}{tipo}{numero}
# Settore 1 = trattati, settore 3 = legislazione secondaria, settore 6 = giurisprudenza
# Tipo: R=Regulation, L=Directive, D=Decision, H=Recommendation,
#       F=Framework Decision, B=Legislative Act, E=articolo trattato
#       CJ=Court of Justice, TJ=General Court, FT=Civil Service Tribunal

CELEX_AUTHOR_MAP = {
    # ── il tipo CELEX determina univocamente l'autore ─────────────
    '1': 'Treaty',           # settore 1 = trattati, sempre
    '6': 'Court',            # settore 6 = giurisprudenza, sempre
    'F': 'Council',          # Framework Decision = sempre Consiglio, pre-Lisbona
    'H': 'Commission_other', # Recommendation = quasi sempre Commissione
    'O': 'Commission_other', # Guidelines/Opinion = quasi sempre Commissione
}

CELEX_PROCEDURE_MAP = {
    '1': 'treaty',
    '6': 'judgment',
    'F': 'delegated_implementing',  # Framework Decision pre-Lisbona
    'H': 'recommendation',
    'O': 'recommendation',
}

def infer_from_celex(celex):
    """
    Inferisce author e procedure dal codice CELEX senza testo né API.
    Restituisce (author, procedure) o (None, None) se il pattern non è riconoscibile.

    Pattern CELEX:
      - Settore 1 (trattati):        1{anno}E{numero}   → Treaty
      - Settore 3 (leg. secondaria): 3{anno}{tipo}{num} → vedi mappa
      - Settore 6 (giurisprudenza):  6{anno}CJ{numero}  → Court
    """
    if pd.isna(celex):
        return None, None

    c = str(celex).strip().upper()

    # Settore 1 — trattati
    if c.startswith('1'):
        return 'Treaty', 'treaty'

    # Settore 6 — giurisprudenza
    if c.startswith('6'):
        return 'Court', 'judgment'

    # Settore 3 — legislazione secondaria
    if c.startswith('3') and len(c) >= 5:
        # Pattern: 3{4_cifre_anno}{tipo_lettera/e}{numero}
        type_char = c[5] if len(c) > 5 else None  # es. 32019R0452 → 'R'

        # Caso speciale: Commission delegated/implementing
        # CELEX delegated: tipo 'R' con prefisso 'D' nel numero es. 32019R2088 no,
        # si riconosce dal titolo — non inferibile solo dal CELEX puro
        # Usiamo il tipo grezzo

        author    = CELEX_AUTHOR_MAP.get(type_char)
        procedure = CELEX_PROCEDURE_MAP.get(type_char)

        return author, procedure

    return None, None


# ── Applica CELEX inference ai nodi ancora senza author o procedure ───────────
print("Applicazione CELEX inference...")

celex_results = nodes['Label'].apply(infer_from_celex)
nodes['celex_author']    = [r[0] for r in celex_results]
nodes['celex_procedure'] = [r[1] for r in celex_results]

# Merge a tre livelli: regex → LLM → CELEX inference
nodes['author']    = (nodes['rx_author']
                      .fillna(nodes['llm_author'])
                      .fillna(nodes['celex_author']))

nodes['procedure'] = (nodes['rx_procedure']
                      .fillna(nodes['llm_procedure'])
                      .fillna(nodes['celex_procedure']))

# tfeu_article e primary_obligation: CELEX non aggiunge info qui
# rimangono regex → LLM come prima

# ── Tracciabilità: fonte di ogni valore di author ─────────────────────────────
# 'regex'   → deterministico, alta affidabilità
# 'llm'     → LLM su testo breve, affidabilità media (non validata su campione)
# 'celex' → inferito dal codice CELEX, affidabile solo per:
#           - Trattati (settore 1): sempre corretto
#           - Giurisprudenza (settore 6): sempre corretto
#           - Framework Decision (F): sempre Consiglio pre-Lisbona
#           - Recommendation (H) e Guidelines (O): quasi sempre Commissione
#           Per R, L, D, B: non inferibile → lasciato None
# 'missing' → nessuna fonte disponibile, escluso dal clustering
def get_author_source(row):
    if pd.notna(row.get('rx_author')):   return 'regex'
    if pd.notna(row.get('llm_author')):  return 'llm'
    if pd.notna(row.get('celex_author')): return 'celex'
    return 'missing'

nodes['author_source'] = nodes.apply(get_author_source, axis=1)

print("Distribuzione fonte author:")
print(nodes['author_source'].value_counts().to_string())
print()

print("\nCopertura feature funzionali (dopo CELEX inference):")
for col in ['author', 'procedure', 'tfeu_article', 'primary_obligation']:
    n = nodes[col].notna().sum()
    print(f"  {col:<25} {n:>5} / {len(nodes)}  ({n/len(nodes)*100:.1f}%)")

print("\nDistribuzione author finale:")
print(nodes['author'].value_counts(dropna=False).to_string())

# Verifica: quanti nodi erano unclassifiable e ora hanno author?
if 'raw_cluster' in nodes.columns:
    prev_missing = nodes[nodes['author'].notna() &
                         (nodes['text_status'] != 'ok')]
    print(f"\nNodi senza testo ma con author inferito da CELEX: {len(prev_missing)}")

Applicazione CELEX inference...
Distribuzione fonte author:
author_source
regex      2214
llm        1527
celex       486
missing     189


Copertura feature funzionali (dopo CELEX inference):
  author                     4227 / 4416  (95.7%)
  procedure                  4227 / 4416  (95.7%)
  tfeu_article               1851 / 4416  (41.9%)
  primary_obligation         3641 / 4416  (82.5%)

Distribuzione author finale:
author
Council                    1204
Commission_other            758
EP_Council                  684
Commission_implementing     501
Treaty                      491
Commission_delegated        268
Court                       221
NaN                         189
Other                       100


## 5. Embedding Semantici su Testo Breve

Solo titolo + prime 200 caratteri del preambolo. Questo testo contiene
la firma funzionale dell'atto (chi adotta, con quale procedura) senza
aggiungere contenuto tematico che disturberebbe il clustering.

In [39]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

print("Caricamento modello sentence-transformers (all-mpnet-base-v2)...")
model = SentenceTransformer(EMBEDDING_MODEL)

def build_short_text(title, preamble):
    t = str(title)    if pd.notna(title)    else ''
    p = str(preamble) if pd.notna(preamble) else ''
    combined = f"{t} {p[:200]}".strip()
    return combined if len(combined) > 10 else None

nodes['short_text'] = nodes.apply(lambda r: build_short_text(r.get('title'), r.get('preamble')), axis=1)

has_text   = nodes['short_text'].notna()
texts      = nodes.loc[has_text, 'short_text'].tolist()
text_idx   = nodes.index[has_text].tolist()

print(f"Nodi con testo: {len(texts)} / {len(nodes)}")
print("Calcolo embedding...")

embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

# Flag: 1.0 se il nodo ha embedding reale, 0.0 se sarà azzerato.
# Viene aggiunto come feature binaria per segnalare al UMAP quali nodi
# hanno un segnale semantico autentico (invece di usare imputazione media,
# che attrarre artificialmente i nodi senza testo verso il centroide).
nodes['has_embedding'] = 0.0
nodes.loc[has_text, 'has_embedding'] = 1.0

# Matrice completa: righe con testo → embedding reale, righe senza → zero.
# Zero-imputation è preferibile a mean-imputation: non distorce la distribuzione
# degli embedding verso il centroide artificiale degli atti con testo.
emb_matrix = np.zeros((len(nodes), embeddings.shape[1]))
for i, idx in enumerate(text_idx):
    emb_matrix[idx] = embeddings[i]

# PCA 50d — solo sui nodi con embedding reale per evitare che gli zeri
# distorcano la direzione dei componenti principali.
pca = PCA(n_components=50, random_state=42)
pca.fit(emb_matrix[nodes['has_embedding'].values == 1.0])
emb_pca = pca.transform(emb_matrix)   # proietta tutto (inclusi gli zeri)

# Azzera le righe dei nodi senza testo nello spazio PCA
# (la proiezione di uno zero-vector può produrre valori residui non nulli
#  a causa del centering interno di PCA)
emb_pca[nodes['has_embedding'].values == 0.0] = 0.0

print(f"Nodi con embedding reale: {int(nodes['has_embedding'].sum())} / {len(nodes)}")
print(f"Shape dopo PCA: {emb_pca.shape}")

Caricamento modello sentence-transformers (all-mpnet-base-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Nodi con testo: 3741 / 4416
Calcolo embedding...


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Nodi con embedding reale: 3741 / 4416
Shape dopo PCA: (4416, 50)


## 5.1 Salvataggio e caricamento embedding
In questo modo non serve lanciare nuovamente la cella di calcolo degli embedding

In [40]:
# ── Salvataggio checkpoint embedding ─────────────────────────────────────────
# Rieseguire gli embedding richiede ~2 minuti e il download del modello.
# Questo checkpoint permette di saltare la cella 5 nelle esecuzioni successive.

emb_checkpoint_file = os.path.join(output_path, 'embeddings_checkpoint.npz')

np.savez(
    emb_checkpoint_file,
    emb_pca       = emb_pca,
    has_embedding = nodes['has_embedding'].values,
    node_ids      = nodes['Id'].values,   # per verificare che i nodi siano gli stessi
)
print(f"Checkpoint embedding salvato: {emb_checkpoint_file}")
print(f"  Shape emb_pca:      {emb_pca.shape}")
print(f"  Nodi con embedding: {int(nodes['has_embedding'].sum())}")

Checkpoint embedding salvato: ..\data\output\golden_power\embeddings_checkpoint.npz
  Shape emb_pca:      (4416, 50)
  Nodi con embedding: 3741


In [41]:
# ── Caricamento checkpoint embedding (se disponibile) ────────────────────────
# Se il checkpoint esiste, salta il calcolo degli embedding (~2 minuti).
# Per forzare il ricalcolo: cancella embeddings_checkpoint.npz oppure
# imposta FORCE_RECOMPUTE = True

FORCE_RECOMPUTE      = False
emb_checkpoint_file  = os.path.join(output_path, 'embeddings_checkpoint.npz')

if not FORCE_RECOMPUTE and os.path.exists(emb_checkpoint_file):
    print("Checkpoint embedding trovato — caricamento in corso...")
    ckpt = np.load(emb_checkpoint_file, allow_pickle=True)

    # Verifica che i nodi siano gli stessi (ordine e ID)
    if np.array_equal(ckpt['node_ids'], nodes['Id'].values):
        emb_pca                = ckpt['emb_pca']
        nodes['has_embedding'] = ckpt['has_embedding']
        print(f"  Caricato: emb_pca shape = {emb_pca.shape}")
        print(f"  Nodi con embedding reale: {int(nodes['has_embedding'].sum())}")
        print("  Salta il calcolo — vai direttamente alla cella 6.")
    else:
        print("[ATTENZIONE] I nodi nel checkpoint non corrispondono al dataframe attuale.")
        print("  Probabilmente hai aggiunto/rimosso nodi — ricalcolo necessario.")
        print("  Imposta FORCE_RECOMPUTE = True per forzare il ricalcolo.")
        FORCE_RECOMPUTE = True  # forza il ricalcolo nella stessa esecuzione

if FORCE_RECOMPUTE or not os.path.exists(emb_checkpoint_file):
    print("Calcolo embedding da zero...")
    # ... qui va tutto il codice originale della cella 5

Checkpoint embedding trovato — caricamento in corso...
  Caricato: emb_pca shape = (4416, 50)
  Nodi con embedding reale: 3741
  Salta il calcolo — vai direttamente alla cella 6.


## 6. Costruzione Matrice Feature Completa

Combina le tre famiglie di feature con pesi differenziati.

Feature strutturali (×2): 9 feature tipate, normalizzate temporalmente, con HITS e k-core  
Feature funzionali (×2): `author`, `procedure` (one-hot)  
`primary_obligation_regex` (×2): deterministico, copertura 1.4%  
`primary_obligation_llm` (×0.5): LLM, copertura ~60%, segnale soft  
TFEU bucket (×1.5): bucketing dell'articolo TFUE di base giuridica  
`has_technical_annex` (×1): flag allegati tecnici  
`has_embedding` (×1): segnala se il nodo ha embedding semantico reale  
Embedding semantici PCA-50 (×1): `all-mpnet-base-v2` su titolo + 200c preambolo

In [42]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

# ── Strutturali ───────────────────────────────────────────────────────────────
# Feature selezionate per minimizzare ridondanza e massimizzare segnale gerarchico.
# Le versioni _norm rimuovono il bias temporale (atti vecchi hanno più citazioni).
struct_feature_cols = [
    # ── Normalizzati temporalmente ─────────────────────────────────────
    'indegree_norm',                  # quante citazioni ricevi, confrontato con atti dello stesso anno
    'pagerank_norm',                  # quanto sei "importante" nella rete, confrontato con atti dello stesso anno
    'hits_authority_norm',            # quanto sei puntato da atti che a loro volta puntano a molti altri, norm. per anno
    'hierarchical_authority_norm',    # quanti atti ti usano come base giuridica (BASED_ON + IMPLEMENTS), norm. per anno

    # ── Ratio (direzione relativa) ─────────────────────────────────────
    'hits_ratio',                     # authority diviso hub: alto = apicale, basso = terminale
    'hierarchical_ratio',             # quanti ti citano come base / quanti citi come base — misura il "peso" gerarchico
    'citation_ratio',                 # indegree diviso outdegree: alto = atto che riceve più di quanto emette

    # ── Topologici ────────────────────────────────────────────────────
    'kcore',                          # quanto sei nel "nucleo" della rete, indipendentemente dal tempo

    # ── Tipate (granulari) ────────────────────────────────────────────
    'based_on_received',              # quanti atti si fondano giuridicamente su di te — segnale apicale più forte
    'implements_received',            # quanti atti ti implementano — distingue L1 (quadro) da L2 (implementazione)
    'amends_received',                # quante volte sei stato modificato — atto "vivo", centrale nel suo settore
    'amends_sent',                    # quanti atti hai modificato — tipico degli atti di revisione tecnica
    'repeals_sent',                   # quanti atti hai abrogato — solo gli atti con autorità alta possono farlo
]
struct_feature_cols = [c for c in struct_feature_cols if c in nodes.columns]
struct_data = nodes[struct_feature_cols].values.astype(float)
# Log1p su feature con distribuzione power-law
for i, col in enumerate(struct_feature_cols):
    if col in ('indegree_norm','pagerank_norm','hits_authority_norm',
               'hierarchical_authority_norm','kcore', 'based_on_received', 
               'implements_received', 'amends_received', 'amends_sent', 'repeals_sent'):
        struct_data[:, i] = np.log1p(struct_data[:, i])
struct_scaled = StandardScaler().fit_transform(struct_data)
print(f"Feature strutturali: {struct_scaled.shape[1]} colonne")

# ── Funzionali (categorici → one-hot) ────────────────────────────────────────
# author e procedure: deterministici (regex) → peso ×2
func_cat_strong = nodes[['author','procedure']].fillna('unknown')
func_encoded_strong = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(func_cat_strong)

# primary_obligation_regex: copertura 1.4% ma completamente deterministico → peso ×2
# Tenuto separato da LLM perché mescolarli nasconde la differenza di affidabilità.
pobl_regex_cat = nodes[['primary_obligation_regex']].fillna('unknown')
pobl_regex_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(pobl_regex_cat)

# primary_obligation_llm: copertura ~60% ma fonte LLM non validata sistematicamente → peso ×0.5
# Contribuisce come segnale soft: orienta il clustering senza dominarlo.
pobl_llm_cat = nodes[['primary_obligation_llm']].fillna('unknown')
pobl_llm_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(pobl_llm_cat)

# TFEU article: bucketing per tipo di competenza
def tfeu_bucket(art):
    if pd.isna(art): return 'unknown'
    a = int(art)
    if   a <= 17:  return 'principles'
    elif a <= 66:  return 'internal_market'
    elif a <= 113: return 'policies'
    elif a <= 197: return 'institutions'
    elif a <= 291: return 'implementing'
    else:          return 'other'

nodes['tfeu_bucket'] = nodes['tfeu_article'].apply(tfeu_bucket)
tfeu_encoded = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit_transform(nodes[['tfeu_bucket']])

annex_feat    = nodes['has_technical_annex'].astype(float).values.reshape(-1,1)
has_emb_feat  = nodes['has_embedding'].values.reshape(-1,1)

# ── Concatenazione con pesi ───────────────────────────────────────────────────
feature_matrix = np.hstack([
    struct_scaled       * 2.0,   # strutturali: segnale gerarchico forte
    func_encoded_strong * 2.0,   # author + procedure: deterministici
    pobl_regex_enc      * 2.0,   # primary_obligation regex: deterministico
    pobl_llm_enc        * 0.5,   # primary_obligation LLM: segnale soft
    tfeu_encoded        * 1.5,
    annex_feat          * 1.0,
    has_emb_feat        * 1.0,
    emb_pca             * 1.0,
])

print(f"Matrice feature: {feature_matrix.shape}")
print(f"  strutturali (x2):           {struct_scaled.shape[1]} col  {struct_feature_cols}")
print(f"  author+procedure (x2):      {func_encoded_strong.shape[1]} col")
print(f"  primary_oblig regex (x2):   {pobl_regex_enc.shape[1]} col  [{nodes['primary_obligation_regex'].notna().sum()} nodi con valore reale]")
print(f"  primary_oblig llm (x0.5):   {pobl_llm_enc.shape[1]} col  [{nodes['primary_obligation_llm'].notna().sum()} nodi con valore reale]")
print(f"  TFEU bucket (x1.5):         {tfeu_encoded.shape[1]} col")
print(f"  annex (x1):                 1 col")
print(f"  has_embedding (x1):         1 col")
print(f"  embedding PCA-50 (x1):      {emb_pca.shape[1]} col")

Feature strutturali: 13 colonne
Matrice feature: (4416, 101)
  strutturali (x2):           13 col  ['indegree_norm', 'pagerank_norm', 'hits_authority_norm', 'hierarchical_authority_norm', 'hits_ratio', 'hierarchical_ratio', 'citation_ratio', 'kcore', 'based_on_received', 'implements_received', 'amends_received', 'amends_sent', 'repeals_sent']
  author+procedure (x2):      17 col
  primary_oblig regex (x2):   5 col  [190 nodi con valore reale]
  primary_oblig llm (x0.5):   7 col  [3505 nodi con valore reale]
  TFEU bucket (x1.5):         7 col
  annex (x1):                 1 col
  has_embedding (x1):         1 col
  embedding PCA-50 (x1):      50 col


## 7. UMAP — Riduzione Dimensionale

In [43]:
import umap

print(f"UMAP: {feature_matrix.shape[1]}d → {UMAP_N_COMPONENTS}d  (può richiedere 2-5 minuti)")

reducer = umap.UMAP(
    n_components=UMAP_N_COMPONENTS, n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST, metric='euclidean', random_state=UMAP_RANDOM_STATE,
)
emb_umap = reducer.fit_transform(feature_matrix)
print(f"Shape dopo UMAP: {emb_umap.shape}")

# Embedding 2D separato per visualizzazione
reducer_2d = umap.UMAP(n_components=2, n_neighbors=UMAP_N_NEIGHBORS,
                        min_dist=0.1, random_state=UMAP_RANDOM_STATE)
emb_2d = reducer_2d.fit_transform(feature_matrix)
nodes['umap_x'] = emb_2d[:, 0]
nodes['umap_y'] = emb_2d[:, 1]
print("Embedding 2D per visualizzazione: ok")

UMAP: 101d → 15d  (può richiedere 2-5 minuti)


c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape dopo UMAP: (4416, 15)


c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Embedding 2D per visualizzazione: ok


## 8. HDBSCAN — Clustering Non Supervisionato

Il numero di cluster (= layer) emerge dai dati.
Gli atti con label `-1` (noise) non appartengono a nessun cluster:
sono strutturalmente anomali e meritano analisi separata.

In [44]:
# ── Separazione nodi con/senza dati funzionali ───────────────────────────────
# I nodi senza author (text_status != 'ok', regex e LLM falliti) non vengono
# clusterizzati: condividono lo stesso profilo non perché hanno lo stesso ruolo
# normativo, ma perché mancano di dati. Includerli creerebbe un cluster
# artefatto. Vengono assegnati a categoria separata (-2) e trattati come
# "unclassifiable" nel report finale — distinti dal noise semantico (-1).
import hdbscan

has_functional = nodes['author'].notna()
nodes_valid    = nodes[has_functional].copy()
nodes_missing  = nodes[~has_functional].copy()

emb_umap_valid = emb_umap[has_functional.values]

print(f"Nodi con dati funzionali:    {len(nodes_valid)} ({len(nodes_valid)/len(nodes)*100:.1f}%)")
print(f"Nodi senza dati funzionali:  {len(nodes_missing)} ({len(nodes_missing)/len(nodes)*100:.1f}%)")
print(f"  → esclusi dal clustering, etichettati come 'unclassifiable'")
print()

# ── Parametri ─────────────────────────────────────────────────────────────────

print(f"HDBSCAN — min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    prediction_data=True,
    cluster_selection_method='eom',
)
cluster_labels = clusterer.fit_predict(emb_umap_valid)

soft_valid     = hdbscan.all_points_membership_vectors(clusterer)
n_real_clusters = soft_valid.shape[1]
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print(f"\nLayer trovati:  {n_clusters}")
print(f"Atti noise:     {n_noise} ({n_noise/len(nodes_valid)*100:.1f}% dei nodi clusterizzati)")
print("\nDistribuzione:")
for lbl, cnt in zip(*np.unique(cluster_labels, return_counts=True)):
    name = 'noise' if lbl == -1 else f'cluster_{lbl}'
    print(f"  {name:<15} {cnt:>5}  ({cnt/len(nodes_valid)*100:.1f}%)")

# ── Ricomposizione dataframe completo ─────────────────────────────────────────
nodes_valid['raw_cluster']   = cluster_labels
nodes_missing['raw_cluster'] = -2   # unclassifiable — dati insufficienti

# Invece di sort_index(), preserva l'ordine esplicito
nodes_valid['_original_order']   = np.where(has_functional.values)[0]
nodes_missing['_original_order'] = np.where(~has_functional.values)[0]

nodes = pd.concat([nodes_valid, nodes_missing])
nodes = nodes.sort_values('_original_order').drop(columns='_original_order')
nodes = nodes.reset_index(drop=True)

# Soft matrix: ricostruisci sull'indice originale
soft = np.zeros((len(nodes), n_real_clusters))
for i, pos in enumerate(np.where(has_functional.values)[0]):
    soft[pos] = soft_valid[i]

print(f"\nSoft membership matrix: {soft.shape}")
print(f"  (righe con tutti zero = nodi unclassifiable: {len(nodes_missing)})")

Nodi con dati funzionali:    4227 (95.7%)
Nodi senza dati funzionali:  189 (4.3%)
  → esclusi dal clustering, etichettati come 'unclassifiable'

HDBSCAN — min_cluster_size=200, min_samples=25

Layer trovati:  6
Atti noise:     197 (4.7% dei nodi clusterizzati)

Distribuzione:
  noise             197  (4.7%)
  cluster_0         278  (6.6%)
  cluster_1         713  (16.9%)
  cluster_2         448  (10.6%)
  cluster_3        1871  (44.3%)
  cluster_4         279  (6.6%)
  cluster_5         441  (10.4%)

Soft membership matrix: (4416, 6)
  (righe con tutti zero = nodi unclassifiable: 189)


## 9. Ordinamento Gerarchico dei Cluster

Ordina i cluster da apicale a terminale usando due segnali:
1. `citation_ratio` medio per cluster (apicale = citato da molti, cita pochi)
2. Citazioni inter-cluster ricevute (apicale = tutti lo citano)

In [45]:
valid = nodes[nodes['raw_cluster'] >= 0].copy()

stats = valid.groupby('raw_cluster').agg(
    n_atti        =('Id','count'),
    mean_indegree =('indegree','mean'),
    mean_outdegree=('outdegree','mean'),
    mean_ratio    =('citation_ratio','mean'),
    mean_pagerank =('pagerank','mean'),
).round(3)

# Flusso citazioni inter-cluster
id_to_cluster = dict(zip(nodes['Id'], nodes['raw_cluster']))
received = {}
for _, edge in edges.iterrows():
    c_src = id_to_cluster.get(edge[src_col], -1)
    c_tgt = id_to_cluster.get(edge[tgt_col], -1)
    if c_src != -1 and c_tgt != -1 and c_src != c_tgt:
        et = edge[type_col] if type_col else 'CITES'
        w  = EDGE_WEIGHTS.get(et, DEFAULT_WEIGHT)
        received[c_tgt] = received.get(c_tgt, 0) + w

stats['received_citations'] = [received.get(c, 0) for c in stats.index]

# Score gerarchico combinato
scaler_mm = MinMaxScaler()
stats['score_ratio']    = scaler_mm.fit_transform(stats[['mean_ratio']])
stats['score_received'] = scaler_mm.fit_transform(stats[['received_citations']])
stats['hierarchy_score'] = stats['score_ratio'] * HIERARCHY_W_RATIO + stats['score_received'] * HIERARCHY_W_RECEIVED

stats = stats.sort_values('hierarchy_score', ascending=False)
stats['layer_rank']  = range(1, len(stats)+1)
stats['layer_label'] = stats['layer_rank'].apply(lambda r: f'L{r}')

print("Cluster ordinati (L1 = più apicale):")
print(stats[['layer_label','n_atti','mean_indegree','mean_outdegree','mean_ratio','received_citations','hierarchy_score']].to_string())

Cluster ordinati (L1 = più apicale):
            layer_label  n_atti  mean_indegree  mean_outdegree  mean_ratio  received_citations  hierarchy_score
raw_cluster                                                                                                    
3                    L1    1871         11.164           7.646       5.713             10043.2         1.000000
4                    L2     279          2.165           4.165       0.906               168.0         0.088237
5                    L3     441          2.170           4.522       0.756               488.0         0.084892
1                    L4     713          1.039           4.976       0.553               154.5         0.049727
0                    L5     278          1.173           4.079       0.532                81.6         0.044554
2                    L6     448          0.837           5.107       0.135                35.3         0.000000


## 10. Calcolo Purezza e Identificazione Atti Patologici

Produce **due misure di patologia complementari**:

| Metrica | Dipende da | Cosa misura |
|---|---|---|
| `is_pathological` | HDBSCAN soft membership | Atto distribuito su più layer nello spazio delle feature |
| `is_pathological_structural` | Feature tipate (BASED_ON) | Atto con tensione apicale+subordinata nelle citazioni |
| `pathology_confidence` | Entrambe | 0=sano, 1=borderline, 2=confermato |

La metrica strutturale è indipendente da `min_cluster_size` e costituisce
il cross-check principale per la robustezza del risultato.

In [46]:
cluster_to_layer = dict(zip(stats.index, stats['layer_label']))
cluster_to_rank  = dict(zip(stats.index, stats['layer_rank']))
real_clusters    = sorted([c for c in set(cluster_labels) if c != -1])

nodes['layer'] = nodes['raw_cluster'].map(cluster_to_layer)
nodes.loc[nodes['raw_cluster'] == -1, 'layer'] = 'noise'           # noise HDBSCAN: dati ok, posizione periferica
nodes.loc[nodes['raw_cluster'] == -2, 'layer'] = 'unclassifiable'  # esclusi a monte: no author
nodes['layer_rank']  = nodes['raw_cluster'].map(cluster_to_rank).fillna(0).astype(int)

# Colonne membership per layer
membership_cols = []
for i, c in enumerate(real_clusters):
    col = f"membership_{cluster_to_layer[c]}"
    nodes[col] = soft[:, i]
    membership_cols.append(col)

# Purity
nodes['purity'] = soft.max(axis=1)

# Entropia di Shannon
def shannon_entropy(row):
    p = row[row > 0]
    return -np.sum(p * np.log2(p)) if len(p) > 0 else 0.0

nodes['layer_entropy'] = [shannon_entropy(soft[i]) for i in range(len(nodes))]

# Level span: distanza tra layer più alto e più basso con membership > soglia
def level_span(i):
    ranks = [cluster_to_rank[c] for j, c in enumerate(real_clusters)
             if soft[i, j] > MEMBERSHIP_THRESHOLD]
    return max(ranks) - min(ranks) if len(ranks) > 1 else 0

nodes['level_span'] = [level_span(i) for i in range(len(nodes))]

# Flag patologico — basato su soft membership HDBSCAN
# ATTENZIONE: questa metrica è sensibile a min_cluster_size.
# Usare in combinazione con is_pathological_structural (sotto).
nodes['is_pathological'] = (
    (nodes['purity'] < PURITY_THRESHOLD) | (nodes['level_span'] >= SPAN_THRESHOLD)
)

# ── Metrica structural_tension ───────────────────────────────────────
# La tensione normativa emerge quando un atto ha CONTEMPORANEAMENTE:
# - alta authority (viene citato come base giuridica da altri)
# - alta subordination (si fonda su basi giuridiche di livello superiore)
# Un atto puramente apicale ha alta authority e bassa subordination → non in tensione.
# Un atto puramente terminale ha bassa authority e alta subordination → non in tensione.
# Un atto ibrido ha entrambe → tensione alta.

apex_feat   = 'hierarchical_authority_norm' if 'hierarchical_authority_norm' in nodes.columns \
              else 'hierarchical_authority'
subord_feat = 'hierarchical_subordination'

apex_scaled   = MinMaxScaler().fit_transform(nodes[[apex_feat]].fillna(0)).flatten()
subord_scaled = MinMaxScaler().fit_transform(nodes[[subord_feat]].fillna(0)).flatten()

# Soglia minima: entrambe le componenti devono essere > 10° percentile
# per evitare che atti con subordination=0.001 e authority=0.999 vengano
# considerati in tensione solo perché il prodotto è > 0.
apex_min   = np.percentile(apex_scaled[apex_scaled > 0],   10)
subord_min = np.percentile(subord_scaled[subord_scaled > 0], 10)

has_both = (apex_scaled >= apex_min) & (subord_scaled >= subord_min)

# Tensione = media armonica delle due componenti (penalizza gli squilibri)
# Solo per gli atti che superano entrambe le soglie minime.
nodes['structural_tension'] = 0.0
nodes.loc[has_both, 'structural_tension'] = (
    2 * apex_scaled[has_both] * subord_scaled[has_both] /
    (apex_scaled[has_both] + subord_scaled[has_both])
)

# Soglia patologia: 75° percentile di chi ha tensione > 0
has_tension = nodes['structural_tension'] > 0
tension_threshold = nodes.loc[has_tension, 'structural_tension'].quantile(0.75)
nodes['is_pathological_structural'] = (
    nodes['structural_tension'] >= tension_threshold
) & has_tension

# pathology_confidence: quante delle due misure di patologia concordano
# 0 = nessuna patologia
# 1 = una sola misura (borderline)
# 2 = entrambe concordano (patologia confermata)
nodes['pathology_confidence'] = (
    nodes['is_pathological'].astype(int) +
    nodes['is_pathological_structural'].astype(int)
)

print(f"Atti con tensione > 0:           {has_tension.sum()} ({has_tension.mean()*100:.1f}%)")
print(f"Soglia patologia (75° pct):      {tension_threshold:.4f}")
print(f"is_pathological_structural:      {nodes['is_pathological_structural'].sum()} ({nodes['is_pathological_structural'].mean()*100:.1f}%)")
print()
print("Structural tension per layer:")
print(nodes[nodes['layer'] != 'unclassifiable'].groupby('layer')['structural_tension']
      .agg(['mean','median','max']).round(4).sort_index().to_string())

# Escludi unclassifiable dalle statistiche — non hanno soft membership reale
nodes_classified = nodes[nodes['layer'] != 'unclassifiable']

print(f"Purezza media:   {nodes_classified['purity'].mean():.3f}")
print(f"Entropia media:  {nodes_classified['layer_entropy'].mean():.3f}")
print()
print("Purezza media per layer:")
print(nodes_classified.groupby('layer')['purity'].agg(['mean','median','min']).round(3).sort_index().to_string())
print()
print(f"is_pathological          (HDBSCAN):    {nodes_classified['is_pathological'].sum():>5} ({nodes_classified['is_pathological'].mean()*100:.1f}%)")
print(f"pathology_confidence == 2 (entrambe):  {(nodes_classified['pathology_confidence']==2).sum():>5} ({(nodes_classified['pathology_confidence']==2).mean()*100:.1f}%)")
print(f"pathology_confidence == 1 (borderline):{(nodes_classified['pathology_confidence']==1).sum():>5} ({(nodes_classified['pathology_confidence']==1).mean()*100:.1f}%)")
print()

Atti con tensione > 0:           276 (6.2%)
Soglia patologia (75° pct):      0.0661
is_pathological_structural:      69 (1.6%)

Structural tension per layer:
         mean  median     max
layer                        
L1     0.0066     0.0  0.2268
L2     0.0000     0.0  0.0000
L3     0.0003     0.0  0.0111
L4     0.0003     0.0  0.0110
L5     0.0000     0.0  0.0000
L6     0.0000     0.0  0.0108
noise  0.0000     0.0  0.0000
Purezza media:   0.764
Entropia media:  0.573

Purezza media per layer:
        mean  median    min
layer                      
L1     0.876   1.000  0.426
L2     0.993   1.000  0.760
L3     0.639   1.000  0.118
L4     0.600   0.704  0.035
L5     0.915   1.000  0.573
L6     0.768   0.872  0.425
noise  0.015   0.007  0.005

is_pathological          (HDBSCAN):     1360 (32.2%)
pathology_confidence == 2 (entrambe):      7 (0.2%)
pathology_confidence == 1 (borderline): 1415 (33.5%)



## 11. Validazione Post-hoc: Composizione Istituzionale

Verifica che l'ordinamento emerso abbia
senso normativo. Se L1 contiene prevalentemente EP_Council/Treaty e l'ultimo
layer contiene prevalentemente Court, l'ordinamento è corretto.

In [49]:
layer_order = [f'L{i}' for i in range(1, n_clusters+1)] + ['noise']

for layer in layer_order:
    sub = nodes[nodes['layer'] == layer]
    if len(sub) == 0: continue
    print(f"{'─'*45}")
    print(f"{layer}  ({len(sub)} atti) — indegree: {sub['indegree'].mean():.1f}  outdegree: {sub['outdegree'].mean():.1f}")
    for author, pct in sub['author'].value_counts(normalize=True).head(4).items():
        bar = '█' * int(pct * 25)
        print(f"  {str(author):<32} {bar} {pct:.0%}")
    print(f"  purezza media: {sub['purity'].mean():.3f}  |  atti patologici: {sub['is_pathological'].sum()} ({sub['is_pathological'].mean()*100:.0f}%)")

─────────────────────────────────────────────
L1  (1871 atti) — indegree: 11.2  outdegree: 7.6
  EP_Council                       █████████ 36%
  Council                          ███████ 28%
  Treaty                           █████ 23%
  Court                            █ 4%
  purezza media: 0.876  |  atti patologici: 387 (21%)
─────────────────────────────────────────────
L2  (279 atti) — indegree: 2.2  outdegree: 4.2
  Commission_delegated             ████████████ 52%
  Commission_other                 ███████████ 44%
  Commission_implementing           3%
  Council                           0%
  purezza media: 0.993  |  atti patologici: 0 (0%)
─────────────────────────────────────────────
L3  (441 atti) — indegree: 2.2  outdegree: 4.5
  Commission_other                 ████████████████ 65%
  Other                            ███ 15%
  Commission_implementing          ██ 9%
  Commission_delegated             █ 6%
  purezza media: 0.639  |  atti patologici: 186 (42%)
──────────────────

## Cella 10b: Nomi interpretativi per layer

In [51]:
# I nomi sono generati UNA SOLA VOLTA e salvati in layer_mapping.csv.
# In esecuzioni successive vengono caricati dal file, non rigenerati.

LAYER_NAMES_FILE = os.path.join(output_path, 'layer_names.json')

def assign_layer_name_llm(layer, sub, edges, id_to_cluster,
                           cluster_to_layer_map,   # ← passato esplicitamente
                           src_col, tgt_col,        # ← passato esplicitamente
                           client):

    author_dist    = sub['author'].value_counts(normalize=True).head(5).to_dict()
    legaltype_dist = sub['LegalType'].value_counts(normalize=True).head(5).to_dict() \
                     if 'LegalType' in sub.columns else {}

    # Top 5 atti più citati — usa CELEX + tipo come fallback al titolo vuoto
    top_cited  = sub.nlargest(5, 'indegree')
    top_titles = []
    for _, row in top_cited.iterrows():
        title = str(row.get('title', '')) if pd.notna(row.get('title')) else ''
        label = row.get('Label', '?')
        ltype = row.get('LegalType', '')
        # Se il titolo è vuoto, usa CELEX + tipo come descrizione minima
        display = title[:120] if len(title) > 5 else f"[{ltype}]"
        top_titles.append(f"- {label}: {display}")

    # Flusso citazioni inter-cluster
    received_from, emitted_to = {}, {}
    for _, edge in edges.iterrows():
        c_src = id_to_cluster.get(edge[src_col], -1)
        c_tgt = id_to_cluster.get(edge[tgt_col], -1)
        if c_src == -1 or c_tgt == -1 or c_src == c_tgt:
            continue
        layer_src = cluster_to_layer_map.get(c_src, '?')
        layer_tgt = cluster_to_layer_map.get(c_tgt, '?')
        if layer_tgt == layer:
            received_from[layer_src] = received_from.get(layer_src, 0) + 1
        if layer_src == layer:
            emitted_to[layer_tgt] = emitted_to.get(layer_tgt, 0) + 1

    prompt = f"""You are an expert in EU law and legislative drafting.
Analyze the following cluster of EU legal acts and assign it a SHORT interpretive name (3-6 words, in English).
The name must reflect the NORMATIVE ROLE of the acts in this cluster, not their subject matter.

CLUSTER STATISTICS:
- Layer rank: {layer} (L1 = most apex, higher = more terminal)
- Number of acts: {len(sub)}
- Mean indegree (citations received): {sub['indegree'].mean():.2f}
- Mean outdegree (citations emitted): {sub['outdegree'].mean():.2f}
- Citation ratio (indegree/outdegree): {sub['citation_ratio'].mean():.2f}
- Mean pagerank: {sub['pagerank'].mean():.6f}
- Purity (HDBSCAN): {sub['purity'].mean():.3f}
- Pathological acts: {sub['is_pathological'].mean()*100:.1f}%

INSTITUTIONAL COMPOSITION:
{chr(10).join(f'- {k}: {v:.0%}' for k,v in author_dist.items())}

LEGAL TYPE DISTRIBUTION:
{chr(10).join(f'- {k}: {v:.0%}' for k,v in legaltype_dist.items())}

CITATION FLOWS (inter-cluster):
- Receives citations from: {dict(sorted(received_from.items(), key=lambda x: -x[1]))}
- Emits citations to:      {dict(sorted(emitted_to.items(), key=lambda x: -x[1]))}

TOP 5 MOST CITED ACTS IN THIS CLUSTER:
{chr(10).join(top_titles) if top_titles else '- (no titles available)'}

TYPED CITATION SIGNALS (mean per act):
{chr(10).join(f'- {feat}: {sub[feat].mean():.2f}' for feat in ['based_on_received','implements_received','amends_received','amends_sent','repeals_sent'] if feat in sub.columns)}

Respond ONLY with this JSON, no extra text:
{{
  "name": "short interpretive name in English (3-6 words)",
  "rationale": "one sentence explaining why this name fits the normative role"
}}"""

    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {'role': 'system', 'content': 'You are an expert in EU legislative drafting. Respond only with valid JSON.'},
                    {'role': 'user',   'content': prompt}
                ],
                temperature=0, max_tokens=200,
            )
            content = resp.choices[0].message.content.strip()
            content = content.replace('```json','').replace('```','').strip()
            parsed  = json.loads(content)
            return parsed.get('name', f'Layer {layer}'), parsed.get('rationale', '')
        except json.JSONDecodeError:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(1)
        except Exception as e:
            print(f"  ERRORE {layer}: {e}")
            return f'[FAILED] Layer {layer}', ''   # ← segnale visibile

    return f'[FAILED] Layer {layer}', ''


# ── Genera O carica nomi (non rigenerare se già salvati) ──────────────────────
if os.path.exists(LAYER_NAMES_FILE):
    print(f"Nomi layer trovati in {LAYER_NAMES_FILE} — caricamento (non rigenerati).")
    with open(LAYER_NAMES_FILE) as f:
        saved = json.load(f)
    auto_names     = saved['names']
    auto_rationale = saved['rationale']
else:
    print("Generazione nomi layer via LLM (verrà fatto una sola volta)...\n")
    auto_names, auto_rationale = {}, {}
    for layer in [f'L{i}' for i in range(1, n_clusters+1)]:
        sub = nodes[nodes['layer'] == layer]
        if len(sub) == 0: continue
        name, rationale = assign_layer_name_llm(
            layer, sub, edges, id_to_cluster,
            cluster_to_layer,   # variabile già definita in cella 9
            src_col, tgt_col,
            client
        )
        auto_names[layer]     = name
        auto_rationale[layer] = rationale
        print(f"  {layer} → {name}")
        print(f"       {rationale}\n")

    # Salva per tutte le esecuzioni future
    with open(LAYER_NAMES_FILE, 'w') as f:
        json.dump({'names': auto_names, 'rationale': auto_rationale}, f, indent=2)
    print(f"\nNomi salvati in {LAYER_NAMES_FILE}")

nodes['layer_name'] = nodes['layer'].map(auto_names).fillna(nodes['layer'])

# Aggiungi layer_name al dataframe e risalva il CSV
nodes['layer_name'] = nodes['layer'].map(auto_names).fillna(nodes['layer'])

Generazione nomi layer via LLM (verrà fatto una sola volta)...

  L1 → Constitutional and Procedural Framework Legislation
       This name fits because the cluster's high layer rank (L1), high citation ratio (5.71), significant Treaty content (23%), and the presence of foundational acts like Regulation 182/2011 (comitology) and core data protection regulations indicate it establishes the fundamental legal principles, institutional procedures, and overarching rules that structure the EU legal order and upon which more specific legislation depends.

  L2 → Commission Technical Implementation Rules
       This name reflects the cluster's normative role as predominantly Commission-adopted regulations and decisions (98% Commission acts, 73% Regulations) that provide detailed technical implementation of higher-level legislation, evidenced by high outdegree citations to L1 apex acts and minimal citation signals for amending or repealing functions.

  L3 → Technical Implementation and Amendme

## 12. Export

In [52]:
drop_cols = [c for c in nodes.columns 
             if c.startswith('rx_') or c.startswith('llm_')
             or c in ('short_text', 'raw_cluster', 'tfeu_bucket')]
output_df = nodes.drop(columns=drop_cols)
output_df.to_csv(output_file, index=False)

print(f"Salvato: {output_file}")
print(f"  {len(output_df)} righe, {len(output_df.columns)} colonne")
print()
new_cols = ['layer','layer_name','layer_rank','purity','layer_entropy','level_span',
            'is_pathological','is_pathological_structural','pathology_confidence',
            'structural_tension','umap_x','umap_y','author','author_source',
            'procedure','tfeu_article','primary_obligation','has_technical_annex',
            'has_embedding','hierarchical_authority','hierarchical_subordination',
            'hierarchical_ratio','hits_authority','hits_hub'] + membership_cols
print("Colonne aggiunte:")
for c in new_cols:
    if c in output_df.columns: print(f"  {c}")

Salvato: ..\data\output\golden_power\nodes_focal_layers.csv
  4416 righe, 95 colonne

Colonne aggiunte:
  layer
  layer_name
  layer_rank
  purity
  layer_entropy
  level_span
  is_pathological
  is_pathological_structural
  pathology_confidence
  structural_tension
  umap_x
  umap_y
  author
  author_source
  procedure
  tfeu_article
  primary_obligation
  has_technical_annex
  has_embedding
  hierarchical_authority
  hierarchical_subordination
  hierarchical_ratio
  hits_authority
  hits_hub
  membership_L5
  membership_L4
  membership_L6
  membership_L1
  membership_L2
  membership_L3


In [53]:
# 10 atti rappresentativi per layer
# Usa i più citati (indegree alto) perché sono quelli che "danno il tono" al layer

for layer in [f'L{i}' for i in range(1, n_clusters+1)] + ['noise']:
    if layer == 'noise':
        # Solo i noise con dati funzionali — escludi chi è finito qui per mancanza di testo
        sub = nodes[(nodes['layer'] == 'noise') & (nodes['author'].notna())]
    else:
        sub = nodes[nodes['layer'] == layer]
    
    if len(sub) == 0:
        continue
    
    nome = auto_names.get(layer, layer)
    print(f"\n{'═'*70}")
    print(f"{layer} — {nome}  ({len(sub)} atti, di cui {nodes[nodes['layer']=='noise']['author'].isna().sum() if layer=='noise' else 0} esclusi per dati mancanti)")
    print(f"{'═'*70}")
    
    campione = sub.nlargest(10, 'indegree')
    
    for _, row in campione.iterrows():
        label   = row.get('Label', '?')
        ltype   = row.get('LegalType', '?')
        year    = row.get('Year', '?')
        author  = row.get('author', '?')
        indeg   = row.get('indegree', 0)
        purity  = row.get('purity', 0)
        tension = row.get('structural_tension', 0)
        title   = str(row.get('title', '')) if pd.notna(row.get('title')) else ''
        
        print(f"  {label:<20} {str(ltype):<20} {year}  indeg={indeg:.0f}  purity={purity:.2f}  tension={tension:.3f}")
        print(f"  autore: {author}")
        if title:
            print(f"  titolo: {title[:100]}")
        print()


══════════════════════════════════════════════════════════════════════
L1 — Constitutional and Procedural Framework Legislation  (1871 atti, di cui 0 esclusi per dati mancanti)
══════════════════════════════════════════════════════════════════════
  12016M005            Treaty               2016.0  indeg=298  purity=1.00  tension=0.000
  autore: Treaty

  32011R0182           Regulation           2011.0  indeg=276  purity=0.94  tension=0.000
  autore: EP_Council
  titolo: REGULATION (EU) No 182/2011 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 16 February 2011 laying

  32016R0679           Regulation           2016.0  indeg=241  purity=1.00  tension=0.000
  autore: EP_Council
  titolo: REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 27 April 2016 on the prote

  12016ETXT            Treaty               2016.0  indeg=230  purity=1.00  tension=0.000
  autore: Treaty

  32018R1725           Regulation           2018.0  indeg=219  purity=0.88  tension=0.071

In [54]:
import pandas as pd

nodes = pd.read_csv('../data/output/golden_power/nodes_focal_layers.csv')

seed_celex = [
    '32019R0452',
    '32021R0821',
    '32008L0114',
    '32022L2557',
    '12016E063',
    '12016E065',
]

print(f"{'CELEX':<20} {'Layer':<15} {'Purity':<10} {'Layer Name'}")
print('─' * 65)

for celex in seed_celex:
    match = nodes[nodes['Id'] == celex]
    if match.empty:
        print(f"{celex:<20} *** NON TROVATO NEL CSV ***")
    else:
        row = match.iloc[0]
        layer      = row.get('layer', '?')
        purity     = row.get('purity', float('nan'))
        layer_name = row.get('layer_name', '')
        print(f"{celex:<20} {str(layer):<15} {purity:<10.3f} {layer_name}")

CELEX                Layer           Purity     Layer Name
─────────────────────────────────────────────────────────────────
32019R0452           L1              0.725      Constitutional and Procedural Framework Legislation
32021R0821           L1              1.000      Constitutional and Procedural Framework Legislation
32008L0114           L1              0.916      Constitutional and Procedural Framework Legislation
32022L2557           L1              1.000      Constitutional and Procedural Framework Legislation
12016E063            L1              1.000      Constitutional and Procedural Framework Legislation
12016E065            L1              1.000      Constitutional and Procedural Framework Legislation
